<a href="https://colab.research.google.com/github/rozaxa/Artificial-Intelligence-Workshop-II/blob/optimizer_comparison/optimization_improvements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
pip install optuna

In [45]:
pip install category_encoders


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 2.0 MB/s eta 0:00:00


In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
import optuna

from sklearn.model_selection import cross_val_score
import category_encoders as ce



# **Regression problem**


In [56]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

X = pd.DataFrame(data, columns=[
    "CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"
])
y = pd.Series(target, name="MEDV")

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [24]:
#Random forest regressor

baseline_model = RandomForestRegressor(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Radom forest Regression RMSE:", baseline_rmse)


Radom forest Regression RMSE: 2.8109631609391226


In [25]:
#GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

In [26]:
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [5, 10, 20], 'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error', verbose=1)

In [27]:
best_grid_params = grid_search.best_params_
best_grid_model = RandomForestRegressor(
    n_estimators=best_grid_params["n_estimators"],
    max_depth=best_grid_params["max_depth"],
    min_samples_split=best_grid_params["min_samples_split"],
    min_samples_leaf=best_grid_params["min_samples_leaf"],
    random_state=42
)


In [28]:
best_grid_model.fit(X_train, y_train)
y_pred_grid = best_grid_model.predict(X_test)
grid_rmse = np.sqrt(mean_squared_error(y_test, y_pred_grid))
print("GridSearchCV Optimized Regression RMSE:", grid_rmse)

GridSearchCV Optimized Regression RMSE: 2.724436890396961


In [29]:
#Optuna

def objective_regression(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    reg_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    reg_model.fit(X_train, y_train)
    y_pred = reg_model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))



In [30]:
study = optuna.create_study(direction="minimize")
study.optimize(objective_regression, n_trials=100)
best_params = study.best_params


[I 2024-12-15 15:48:21,778] A new study created in memory with name: no-name-6a04187b-05a0-4dde-be66-6743ec0a51f0
[I 2024-12-15 15:48:22,018] Trial 0 finished with value: 3.133854130522907 and parameters: {'n_estimators': 60, 'max_depth': 17, 'min_samples_split': 3, 'min_samples_leaf': 3}. Best is trial 0 with value: 3.133854130522907.
[I 2024-12-15 15:48:22,484] Trial 1 finished with value: 3.3832592803198245 and parameters: {'n_estimators': 140, 'max_depth': 28, 'min_samples_split': 8, 'min_samples_leaf': 4}. Best is trial 0 with value: 3.133854130522907.
[I 2024-12-15 15:48:23,186] Trial 2 finished with value: 3.438149369282391 and parameters: {'n_estimators': 253, 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 0 with value: 3.133854130522907.
[I 2024-12-15 15:48:23,568] Trial 3 finished with value: 3.3928111086434036 and parameters: {'n_estimators': 124, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 4}. Best is trial 0 with value: 3.133

In [31]:
best_params = study.best_params
optimized_model = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=42
)
optimized_model.fit(X_train, y_train)
y_pred_optimized = optimized_model.predict(X_test)
optimized_rmse = np.sqrt(mean_squared_error(y_test, y_pred_optimized))
print("Optuna Optimized Regression RMSE:", optimized_rmse)

Optimized Regression RMSE: 2.6571075857621977
Baseline Regression RMSE: 2.8109631609391226


In [32]:
print("Baseline Regression RMSE:", baseline_rmse)
print("GridSearchCV Optimized Regression RMSE:", grid_rmse)
print("Optuna Optimized Regression RMSE:", optimized_rmse)

Baseline Regression RMSE: 2.8109631609391226
GridSearchCV Optimized Regression RMSE: 2.724436890396961
Optuna Optimized Regression RMSE: 2.6571075857621977


# **Classification problem**

In [50]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]
data = pd.read_csv(url, header=None, names=columns, na_values="?")
data.dropna(inplace=True)



In [51]:
label_enc = LabelEncoder()
data["income"] = label_enc.fit_transform(data["income"])
X = pd.get_dummies(data.drop("income", axis=1), drop_first=True)
y = data["income"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
default_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
default_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [36]:
y_pred_default = default_model.predict(X_test)
default_accuracy = accuracy_score(y_test, y_pred_default)
print("Default accurancy:", default_accuracy)

Default accurancy: 0.8582834331337326


In [38]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
best_grid_params = grid_search.best_params_
print("Best GridSearchCV parameters:", best_grid_params)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best GridSearchCV parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}


In [39]:
best_grid_model = RandomForestClassifier(**best_grid_params, random_state=42)
best_grid_model.fit(X_train, y_train)
y_pred_grid = best_grid_model.predict(X_test)
grid_accuracy = accuracy_score(y_test, y_pred_grid)
print("GridSearchCV Optimized accuracy:", grid_accuracy)

GridSearchCV Optimized accuracy: 0.8667280822969445


In [44]:
def objective_classification(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    scores = cross_val_score(clf, X_train, y_train, cv=3, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective_classification, n_trials=100)

best_optuna_params = study.best_params
print("Best Optuna parameters:", best_optuna_params)

optimized_model = RandomForestClassifier(
    n_estimators=best_optuna_params["n_estimators"],
    max_depth=best_optuna_params["max_depth"],
    min_samples_split=best_optuna_params["min_samples_split"],
    min_samples_leaf=best_optuna_params["min_samples_leaf"],
    random_state=42
)
optimized_model.fit(X_train, y_train)
y_pred_optimized = optimized_model.predict(X_test)
optimized_accuracy = accuracy_score(y_test, y_pred_optimized)

print("Optuna Optimized accuracy:", optimized_accuracy)

[I 2024-12-15 16:14:38,307] A new study created in memory with name: no-name-3f94a533-5cee-47b4-8180-f58aba28eaef
[I 2024-12-15 16:14:53,567] Trial 0 finished with value: 0.8604116889944331 and parameters: {'n_estimators': 236, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.8604116889944331.
[I 2024-12-15 16:15:01,076] Trial 1 finished with value: 0.8561886880049188 and parameters: {'n_estimators': 164, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8604116889944331.
[I 2024-12-15 16:15:09,866] Trial 2 finished with value: 0.8397574737079815 and parameters: {'n_estimators': 279, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.8604116889944331.
[I 2024-12-15 16:15:22,843] Trial 3 finished with value: 0.8609875047300019 and parameters: {'n_estimators': 159, 'max_depth': 29, 'min_samples_split': 9, 'min_samples_leaf': 1}. Best is trial 3 with value:

Best Optuna parameters: {'n_estimators': 284, 'max_depth': 28, 'min_samples_split': 9, 'min_samples_leaf': 2}
Optuna Optimized accuracy: 0.8687240902809765


In [41]:
print("Default accuracy:", default_accuracy)
print("GridSearchCV Optimized accuracy:", grid_accuracy)
print("Optuna Optimized accuracy:", optimized_accuracy)

Default accuracy: 0.8582834331337326
GridSearchCV Optimized accuracy: 0.8667280822969445
Optuna Optimized accuracy: 0.8694917856594503
